In [1]:

# 1. IMPORT LIBRARIES

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, hamming_loss




# 2. SIMPLE PREPROCESSING


stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    words = word_tokenize(text.lower())
    words = [lemmatizer.lemmatize(w) for w in words
             if w.isalpha() and w not in stop_words]
    return " ".join(words)



# 3. SAMPLE DATA


data = [
    ("Acute myocardial infarction with chest pain",
     ["Cardio", "MI"]),
    
    ("Irregular heartbeat and arrhythmia",
     ["Cardio", "Arrhythmia"]),
    
    ("High blood sugar and diabetes",
     ["Endocrine", "Diabetes"]),
]

texts = [clean_text(d[0]) for d in data]
labels = [d[1] for d in data]


# 4. MULTI-LABEL ENCODING


mlb = MultiLabelBinarizer()
Y = mlb.fit_transform(labels)

label_index = {label: i for i, label in enumerate(mlb.classes_)}


# 5. TF-IDF VECTORIZATION


vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(texts)


# 6. TRAIN MODEL


X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.3, random_state=1
)

model = OneVsRestClassifier(LogisticRegression(max_iter=1000))
model.fit(X_train, Y_train)

Y_pred = model.predict(X_test)



# 7. SIMPLE HIERARCHY RULE


# If MI predicted → Cardio must be predicted
if "MI" in label_index and "Cardio" in label_index:
    mi_idx = label_index["MI"]
    cardio_idx = label_index["Cardio"]
    
    for row in Y_pred:
        if row[mi_idx] == 1:
            row[cardio_idx] = 1


# 8. EVALUATION METRICS


micro = f1_score(Y_test, Y_pred, average="micro")
macro = f1_score(Y_test, Y_pred, average="macro")
hamming = hamming_loss(Y_test, Y_pred)

print("Micro F1 Score :", round(micro, 3))
print("Macro F1 Score :", round(macro, 3))
print("Hamming Loss   :", round(hamming, 3))

Micro F1 Score : 0.0
Macro F1 Score : 0.0
Hamming Loss   : 0.4


c:\Users\Shaarukesh\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\multiclass.py:90: UserWarning: Label not 4 is present in all training examples.
  warnings.warn(
c:\Users\Shaarukesh\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [2]:
# 1. IMPORT LIBRARIES

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, hamming_loss


# 2. PREPROCESSING

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    words = word_tokenize(text.lower())
    words = [lemmatizer.lemmatize(w) for w in words
             if w.isalpha() and w not in stop_words]
    return " ".join(words)


# 3. EXPANDED DATASET (improves learning)

data = [
    ("Acute myocardial infarction with chest pain", ["Cardio","MI"]),
    ("Heart attack symptoms and severe chest pressure", ["Cardio","MI"]),
    ("Irregular heartbeat and arrhythmia", ["Cardio","Arrhythmia"]),
    ("Patient suffering from cardiac arrhythmia", ["Cardio","Arrhythmia"]),
    ("High blood sugar and diabetes", ["Endocrine","Diabetes"]),
    ("Chronic diabetes with elevated glucose", ["Endocrine","Diabetes"]),
]

texts = [clean_text(d[0]) for d in data]
labels = [d[1] for d in data]


# 4. MULTI-LABEL ENCODING

mlb = MultiLabelBinarizer()
Y = mlb.fit_transform(labels)

label_index = {label: i for i, label in enumerate(mlb.classes_)}


# 5. BETTER TF-IDF FEATURES

vectorizer = TfidfVectorizer(
    ngram_range=(1,2),      # captures phrases like "chest pain"
    min_df=1,
    max_df=0.9
)

X = vectorizer.fit_transform(texts)


# 6. TRAIN MODEL

X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.3, random_state=42
)

model = OneVsRestClassifier(
    LogisticRegression(max_iter=2000, class_weight="balanced")
)

model.fit(X_train, Y_train)

Y_pred = model.predict(X_test)


# 7. HIERARCHY RULE

if "MI" in label_index and "Cardio" in label_index:
    mi_idx = label_index["MI"]
    cardio_idx = label_index["Cardio"]

    for row in Y_pred:
        if row[mi_idx] == 1:
            row[cardio_idx] = 1


# 8. EVALUATION

micro = f1_score(Y_test, Y_pred, average="micro")
macro = f1_score(Y_test, Y_pred, average="macro")
hamming = hamming_loss(Y_test, Y_pred)

print("Micro F1 Score :", round(micro,3))
print("Macro F1 Score :", round(macro,3))
print("Hamming Loss   :", round(hamming,3))

Micro F1 Score : 0.0
Macro F1 Score : 0.0
Hamming Loss   : 0.8


c:\Users\Shaarukesh\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\multiclass.py:90: UserWarning: Label not 4 is present in all training examples.
  warnings.warn(
c:\Users\Shaarukesh\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [3]:
# 1. IMPORT LIBRARIES

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier


# 2. PREPROCESSING

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    words = word_tokenize(text.lower())
    words = [lemmatizer.lemmatize(w) for w in words
             if w.isalpha() and w not in stop_words]
    return " ".join(words)


# 3. SAMPLE DATA

data = [
    ("Acute myocardial infarction with chest pain",
     ["Cardio","MI"]),
    
    ("Irregular heartbeat and arrhythmia",
     ["Cardio","Arrhythmia"]),
    
    ("High blood sugar and diabetes",
     ["Endocrine","Diabetes"]),
]

texts = [clean_text(d[0]) for d in data]
labels = [d[1] for d in data]


# 4. MULTI-LABEL ENCODING

mlb = MultiLabelBinarizer()
Y = mlb.fit_transform(labels)


# 5. TF-IDF

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(texts)


# 6. TRAIN MODEL

model = OneVsRestClassifier(LogisticRegression(max_iter=1000))
model.fit(X, Y)


# 7. DEFINE HIERARCHY

hierarchy = {
    "Cardio": ["MI", "Arrhythmia"],
    "Endocrine": ["Diabetes"]
}


# 8. FUNCTION TO PRINT HIERARCHY

def print_hierarchy(predicted_labels):

    for parent, children in hierarchy.items():

        if parent in predicted_labels:
            print(parent)

            for child in children:
                if child in predicted_labels:
                    print("   └──", child)


# 9. TEST WITH NEW SENTENCE

test_sentence = "Patient suffering from myocardial infarction and chest pain"

cleaned = clean_text(test_sentence)

X_test = vectorizer.transform([cleaned])

prediction = model.predict(X_test)

predicted_labels = mlb.inverse_transform(prediction)[0]

print("Predicted Labels:", predicted_labels)
print("\nHierarchy Output:")

print_hierarchy(predicted_labels)

Predicted Labels: ('Cardio',)

Hierarchy Output:
Cardio
